<a href="https://colab.research.google.com/github/suyon0416/26-2_ESAA_YB/blob/main/ESAA_YB_WEEK13_%EC%88%98%EC%83%81%EC%9E%91_%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 수상작 리뷰 | March Machine Learning Mania 2025
- 수상자 Mohammad Odeh (modeh7), Kaggle 1위
- 대회 목표 NCAA 남녀 대학 농구 토너먼트에서 각 팀의 승리 확률을 예측한다.
- 평가 지표 Brier Score. 예측 확률과 실제 결과(승리 1, 패배 0)의 차이를 제곱해 평균한 값으로, 낮을수록 좋다.

## 1. 데이터
- MRegularSeasonDetailedResults.csv / WRegularSeasonDetailedResults.csv:	남녀 정규시즌 경기별 상세 기록. 시즌, 팀 ID, 점수, 리바운드, 어시스트, 턴오버 등을 포함한다.
- MNCAATourneyCompactResults.csv / WNCAATourneyCompactResults.csv:	과거 토너먼트 경기 결과. WTeamID와 LTeamID로 승리팀과 패배팀을 확인한다.
- MNCAATourneySeeds.csv / WNCAATourneySeeds.csv:	시즌별 팀의 토너먼트 시드.
- SampleSubmissionStage2.csv:	최종 제출 양식. ID는 시즌_작은팀ID_큰팀ID, Pred는 작은 팀 ID를 가진 팀의 승리 확률이다.

파일명 맨 앞의 M과 W는 각각 남자부와 여자부를 뜻한다.

## 2. 코드 흐름
1. 특성 구성 — raddar의 공개 노트북을 바탕으로 경기 기록에서 만든 팀 평균 통계와 팀 전력 지표를 사용했다.
2. 특성 선택 및 모델 비교 — 최종적으로 29개 특성을 선택했다. CatBoost, LightGBM, XGBoost를 비교했고, 우승자의 실험에서는 XGBoost가 가장 좋은 결과를 냈다.
3. 승리 확률 예측 및 후처리 — XGBoost로 매치업별 확률을 예측한 뒤, 남녀부 확률에 보정 규칙을 적용하고 남자부 일부 경기의 확률은 시드와 전문가 판단을 참고해 직접 조정했다.

## 3. 주요 코드 분석
- XGBoost로 점수 차를 예측하고, 스플라인으로 승리 확률로 변환

In [ ]:
t = 25
dat = list(zip(oof_preds, np.array(oof_targets)>0))
dat = sorted(dat, key = lambda x: x[0])
pred, label = list(zip(*dat))
spline_model = UnivariateSpline(np.clip(pred, -t, t), label, k=5)
spline_fit = np.clip(spline_model(np.clip(oof_preds, -t, t)), 0.01, 0.99)
print(f"brier: {brier_score_loss(np.array(oof_targets)>0, spline_fit)}")

- 점수 차 예측 — 앞선 학습 셀에서 한 시즌을 검증용으로 빼고 나머지 시즌으로 XGBoost를 학습한다. 이 과정을 시즌마다 반복해, 각 경기의 예측 점수 차를 oof_preds에 모은다.
- 승리 확률 변환 — 실제 점수 차가 양수면 첫 번째 팀의 승리(True)로 표시한다. 예측 점수 차를 −25점에서 +25점 사이로 제한한 뒤, 실제 승패와의 관계를 5차 스플라인 곡선으로 학습한다. 변환된 확률은 1%에서 99% 사이로 제한한다.
- 최종 제출값 생성 — 시즌별 모델의 예측을 각각 확률로 변환해 평균한다. 이후 원본 코드의 x + x * 0.1 if x < 0.85 else x로 85% 미만의 확률을 기존 값의 1.1배로 조정한다. 예를 들어 60%는 66%가 된다. 마지막으로 지정된 6경기의 확률을 수동으로 덮어쓴다.

## 4. 새롭게 알게 된 점 / 배운 점
- 경기의 예상 점수 차를 승리 확률로 바꿀 수 있다는 점을 배웠다. 우승자는 XGBoost로 두 팀의 점수 차를 예측한 뒤, 과거 경기의 예측 점수 차와 실제 승패 사이의 관계를 스플라인으로 학습해 승리 확률을 구했다.
- 승리팀을 맞히는 것과 확률을 잘 예측하는 것은 다르다는 점을 배웠다. Brier Score에서는 승리 확률을 높게 준 팀이 패배하면 오차가 커진다. 따라서 우승자의 수동 확률 조정이 다른 시즌에도 효과적인지 판단하려면, 보정 규칙을 정하는 데 사용하지 않은 시즌에서 보정 전후의 점수를 비교해야 한다.